# Chemical equilibrium by Gibbs energy minimisation

*chemistry · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/gibbs_minimisation)

What comes out of a methane–air flame? At fixed $T$ and $P$ the products settle where the Gibbs energy $G = \sum_j n_j \mu_j$ is lowest, subject to conservation of every element, $A_{ij} n_j = b_i$. A batched Newton method (the RAND method) solves 10 species at a whole range of temperatures at once, shows CO₂ and H₂O falling apart into CO, H₂, OH, H and O as the gas gets hotter, and finds the adiabatic flame temperature.

**Vector form**

$$\min_{\mathbf n > 0}\; G(\mathbf n) = \sum_j n_j\Big[\mu^\circ_j(T) + RT\ln\frac{n_j P}{n_{\mathrm{tot}}P^\circ}\Big]\quad\text{s.t.}\quad A\,\mathbf n = \mathbf b,\qquad \frac{\boldsymbol\mu}{RT} = A^{\mathsf T}\boldsymbol\pi,\qquad H_{\mathrm{prod}}(T_{ad}) = H_{\mathrm{react}}$$

**Index (tensor) form**

$$\frac{G_b}{RT_b} = n_{bj}\big(g_{bj} + \ln n_{bj} - \ln n_{\mathrm{tot},b} + \ln P\big),\qquad A_{ij}\,n_{bj} = b_i,\qquad \big[A_{ij}n_{bj}A_{lj}\big]\,\pi_{bl} + \big[A_{ij}n_{bj}\big]\,\Delta\ln n_{\mathrm{tot},b} = r_{bi},\qquad \Delta\ln n_{bj} = -\frac{\mu_{bj}}{RT} + \pi_{bi}A_{ij} + \Delta\ln n_{\mathrm{tot},b}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{ij}$ | `element_matrix(dev)` | (E, S) = (4, 10) | atoms of element $i$ (C, H, O, N) in species $j$ (CO₂, H₂O, CO, H₂, O₂, N₂, OH, H, O, NO) |
| $b_i$ | `feed(phi)` | (F, E) | atoms fed in by CH₄ + $(2/\varphi)$(O₂ + 3.76 N₂), one row per equivalence ratio $\varphi$ |
| $T_b$ | `temperatures(T_min, T_max, n)` | (B,) | the batch: every temperature is an independent minimisation, solved together |
| $g_{bj} = \mu^\circ_j(T_b)/RT_b$ | `standard_potential(T, th)` | (B, S) | $[H_j(T) - T S_j(T)]/RT$ from $\Delta_f H^\circ$, $S^\circ$ and a constant $c_p$ |
| $G_b/RT_b = n_{bj}\mu_{bj}/RT_b$ | `einsum('bj,bj->b', n, mu)` | (B,) | repeated $j$ ⇒ summed: the Gibbs energy of each mixture |
| $A_{ij}n_{bj}$ | `einsum('ij,bj->bi', A, n)` | (B, E) | atoms of each element present; equals $b_i$ at the solution |
| $A_{ij}\,n_{bj}\,A_{lj}$ | `einsum('ij,bj,lj->bil', A, n, A)` | (B, E, E) | $j$ summed: the Newton matrix, a weighted $A\,\mathrm{diag}(n)\,A^{\mathsf T}$ per row |
| $\pi_{bi},\ \Delta\ln n_{\mathrm{tot},b}$ | `torch.linalg.solve(J, rhs)` | (B, E+1) | one small $(E{+}1)\times(E{+}1)$ system per temperature, all solved in one call |
| $\Delta\ln n_{bj}$ | `species_update(A, mu, pi, dln_ntot)` | (B, S) | $-\mu_{bj}/RT + \pi_{bi}A_{ij} + \Delta\ln n_{\mathrm{tot}}$, $i$ summed; damped like NASA CEA |
| $\partial (G/RT)/\partial n_{bj}$ | `torch.autograd.grad(gibbs(n, g, P).sum(), n)` | (B, S) | autograd check: equals $\pi_{bi}A_{ij}$ at the minimum (Lagrange condition) |
| $H_b = n_{bj}\,h_j(T_b)$ | `enthalpy(n, T, th)` | (F,) | product enthalpy; bisection on $T$ until it equals the reactant enthalpy: $T_{ad}$ |

Why minimise $G$? At constant $T$ and $P$ the second law says a closed system can only lower its Gibbs energy, so equilibrium is the lowest point of $G$ that keeps every atom: a constrained minimisation. With Lagrange multipliers $\pi_i$ (one per element) the condition is $\mu_j/RT = \sum_i \pi_i A_{ij}$: each species' chemical potential is the sum of the "element potentials" of its atoms. This replaces a list of equilibrium constants (one per reaction) by one optimisation over all species, which is how codes such as NASA CEA, STANJAN and Cantera compute flames, rocket exhausts and atmospheres. The solver here is the RAND method in the Gordon–McBride form (NASA RP-1311): Newton's method on $\ln n_j$, eliminated down to a small $(E{+}1)\times(E{+}1)$ system for $\pi_i$ and $\Delta\ln n_{tot}$, with the step limited so major species change by at most $e^2$ per step. Working in logarithms keeps trace species of mole fraction $10^{-30}$ positive and exact. The console prints the two optimality checks (element balance and $\mu_j/RT = \pi_i A_{ij}$, both near machine precision) and an autograd check: differentiating $G/RT$ with torch gives back $\pi_i A_{ij}$. What to look for: below about 1500 K the products are just CO₂, H₂O and N₂ (plus O₂ when lean); above 2000 K they dissociate (CO₂ → CO + ½O₂, H₂O → OH + ½H₂, then into atoms), because the entropy term $-T\,S$ wins over the bond enthalpy. Dissociation absorbs heat, so the equilibrium flame is about 100 K cooler than one whose products stayed intact (lower right), and the hottest flame burns slightly rich of stoichiometric. Raise $P$ to see Le Chatelier: fewer molecules are favoured, so dissociation is suppressed. Data: $\Delta_f H^\circ$(298 K) and $S^\circ$(298 K) are rounded values from the NIST-JANAF tables. The heat capacities are illustrative constants (roughly the mean $c_p$ from 298 to 2000 K), not tabulated functions of $T$, so the compositions are semi-quantitative: this model gives $T_{ad} \approx 2260$ K for stoichiometric methane–air at 1 bar, where accurate thermochemistry gives about 2225 K. Simplifications: ideal gases, no condensed phases (no soot, no liquid water), no CH₄ or other hydrocarbons among the products, and equilibrium only: real flames freeze CO and NO by kinetics, not thermodynamics.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/gibbs_minimisation/main.py
"""Main loop: thermochemistry → equilibrium at every temperature → adiabatic flame temperature → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

T_MIN = 1000.0  # K, start of the temperature sweep
T_COLD = 1000.0  # K, "undissociated" reference composition for the frozen flame temperature


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    t_start = time.perf_counter()
    P, phi0 = params["P_bar"], params["phi"]
    A = tensors.element_matrix(dev)  # (E, S)
    th = tensors.thermo(dev)  # each (S,)

    # --- 1) one feed, a batch of temperatures: composition x_j(T) ---
    T = tensors.temperatures(T_MIN, params["T_max"], 30 if fast else 111, dev)  # (B,)
    b = tensors.feed(torch.tensor([phi0], dtype=torch.float64, device=dev))[0]  # (E,)
    sweep = solve.temperature_sweep(T, A, b, P, th, n_iter=80)
    n = sweep["n"]  # (B, S)
    g = equation.standard_potential(T, th)
    G = equation.gibbs(n, g, P)  # (B,) G/RT at the minimum
    balance, kkt = solve.residuals(A, b.expand(len(T), -1), n, sweep["pi"], g, P)

    # autograd check: ∂(G/RT)/∂n_j at the minimum equals μ_j/RT = π_i A_ij (element potentials)
    n_req = n.clone().requires_grad_(True)
    (grad,) = torch.autograd.grad(equation.gibbs(n_req, g, P).sum(), n_req)
    major = equation.mole_fractions(n) > 1e-8
    grad_err = (grad - torch.einsum("bi,ij->bj", sweep["pi"], A))[major].abs().max().item()

    # --- 2) a batch of feeds (equivalence ratios): adiabatic flame temperature ---
    phis = torch.linspace(0.6, 1.5, 7 if fast else 19, dtype=torch.float64, device=dev)  # (F,)
    if not torch.isclose(phis, torch.tensor(phi0, dtype=torch.float64)).any():
        phis = torch.sort(torch.cat([phis, torch.tensor([phi0], dtype=torch.float64, device=dev)])).values
    bF, H_r = tensors.feed(phis), tensors.reactant_enthalpy(phis)  # (F, E), (F,)
    T_ad, n_ad = solve.flame_temperature(A, bF, H_r, P, th)
    cold = solve.equilibrium(equation.standard_potential(torch.full_like(phis, T_COLD), th), A, bF, P)["n"]
    T_frozen = solve.frozen_flame_temperature(cold, H_r, th)
    k0 = int(torch.argmin((phis - phi0).abs()))

    x = equation.mole_fractions(n)
    names = tensors.SPECIES
    print(f"CH4 + {2 / phi0:.3g} (O2 + 3.76 N2)   ·   φ = {phi0}   ·   P = {P} bar   ·   {len(names)} species, "
          f"{len(tensors.ELEMENTS)} elements, {len(T)} temperatures solved as one batch")  # fmt: skip
    print("mole fractions:   " + "".join(f"{s:>9}" for s in names))
    for T_show in (1500.0, 2000.0, 2500.0, 3000.0, 3500.0):
        if T_show <= params["T_max"]:
            i = int(torch.argmin((T - T_show).abs()))
            print(f"  T = {T[i].item():6.0f} K     " + "".join(f"{v:9.2e}" for v in x[i].tolist()))
    print(f"optimality: max |A n - b| = {balance:.1e}   ·   max |μ_j/RT - π_i A_ij| = {kkt:.1e}   ·   "
          f"autograd ∂(G/RT)/∂n_j vs π_i A_ij: {grad_err:.1e}")  # fmt: skip
    print(f"adiabatic flame temperature at φ = {phis[k0].item():.2f}: {T_ad[k0].item():.0f} K with dissociation, "
          f"{T_frozen[k0].item():.0f} K if the products did not dissociate")  # fmt: skip
    k_max = int(torch.argmax(T_ad))
    print(f"hottest flame over φ ∈ [{phis[0].item():.1f}, {phis[-1].item():.1f}]: "
          f"{T_ad[k_max].item():.0f} K at φ = {phis[k_max].item():.2f}   ·   solved in "
          f"{time.perf_counter() - t_start:.1f} s")  # fmt: skip
    x_ad = equation.mole_fractions(n_ad)[k0]  # (S,) at the flame temperature
    print(f"at T_ad (φ = {phis[k0].item():.2f}): x_CO = {x_ad[names.index('CO')].item():.2e}, "
          f"x_NO = {x_ad[names.index('NO')].item():.2e} (equilibrium, no kinetics)")  # fmt: skip

    return plot.animate(
        T, x, G, phis, T_ad, T_frozen, phis[k0].item(), T_ad[k0].item(), names, out_dir / "gibbs_minimisation.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/gibbs_minimisation/equation.py
r"""Physics → tensor translation.

At fixed temperature T and pressure P, a reacting ideal-gas mixture settles where its Gibbs energy is lowest:

    G(n) = Σ_j n_j μ_j,      μ_j = μ°_j(T) + RT ln( n_j P / (n_tot P°) ),      n_tot = Σ_j n_j

subject to element conservation (atoms are neither made nor destroyed):

    A_{ij} n_j = b_i          (j summed: A_{ij} = atoms of element i in species j, b_i = atoms fed in)

Indices: b = batch (one temperature or one equivalence ratio per row), j/k = species, i/l = element.
At the minimum (Lagrange multipliers π_i, the "element potentials"):

    μ_{bj} / RT = π_{bi} A_{ij}       (i summed)

so every species is built from the same per-element chemical potentials. Dimensionless g = μ°/RT throughout.
"""

import math

import torch

from .tensors import T0, R


def standard_potential(T: torch.Tensor, th: dict) -> torch.Tensor:
    r"""g_{bj} = μ°_j(T_b) / RT_b = [H_j(T) - T S_j(T)] / RT with a constant c_p:   T (B,) → (B, S).

    H_j(T) = ΔfH°_j + c_{p,j} (T - T0),     S_j(T) = S°_j + c_{p,j} ln(T / T0)
    """
    T = T[:, None]  # (B, 1) broadcast against species (S,)
    H = 1e3 * th["dH"] + th["cp"] * (T - T0)  # J/mol
    S = th["S"] + th["cp"] * torch.log(T / T0)  # J/(mol K)
    return (H - T * S) / (R * T)


def chemical_potential(n: torch.Tensor, g: torch.Tensor, P: float) -> torch.Tensor:
    r"""μ_{bj}/RT = g_{bj} + ln(n_{bj} / n_tot,b) + ln(P/P°):   (B, S) → (B, S).   P in bar."""
    n_tot = n.sum(-1, keepdim=True)
    return g + torch.log(n / n_tot) + math.log(P)


def chemical_potential_log(ln_n: torch.Tensor, ln_ntot: torch.Tensor, g: torch.Tensor, P: float) -> torch.Tensor:
    """The same μ/RT from ln n_j and ln n_tot: the solver works with logarithms, so trace species of
    mole fraction 1e-40 stay representable."""
    return g + ln_n - ln_ntot[:, None] + math.log(P)


def gibbs(n: torch.Tensor, g: torch.Tensor, P: float) -> torch.Tensor:
    r"""G_b / RT = n_{bj} μ_{bj} / RT   (j summed):   (B, S) → (B,)."""
    return torch.einsum("bj,bj->b", n, chemical_potential(n, g, P))


def element_totals(A: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""A_{ij} n_{bj}   (j summed): atoms of each element present,   (E, S), (B, S) → (B, E)."""
    return torch.einsum("ij,bj->bi", A, n)


def newton_system(A, n, n_tot, mu, b):
    r"""Linearised optimality conditions (the Gordon–McBride form of the RAND method), for unknowns
    π_{bi} and Δln n_tot:

        Σ_l [A_{ij} n_j A_{lj}] π_l + [A_{ij} n_j] Δln n_tot = b_i - A_{ij} n_j + A_{ij} n_j μ_j
        Σ_l [A_{lj} n_j]        π_l + [Σ_j n_j - n_tot] Δln n_tot = n_tot - Σ_j n_j + Σ_j n_j μ_j

    (j summed in every bracket). Returns J (B, E+1, E+1) and rhs (B, E+1).
    """
    M = torch.einsum("ij,bj,lj->bil", A, n, A)  # (B, E, E): j summed
    a = element_totals(A, n)  # (B, E)
    corner = n.sum(-1) - n_tot  # (B,)
    J = torch.cat(
        [torch.cat([M, a[:, :, None]], dim=2), torch.cat([a[:, None, :], corner[:, None, None]], dim=2)], dim=1
    )
    rhs_top = b - a + torch.einsum("ij,bj,bj->bi", A, n, mu)
    rhs_bot = n_tot - n.sum(-1) + torch.einsum("bj,bj->b", n, mu)
    return J, torch.cat([rhs_top, rhs_bot[:, None]], dim=1)


def species_update(A, mu, pi, dln_ntot):
    r"""Δln n_{bj} = -μ_{bj} + π_{bi} A_{ij} + Δln n_tot,b   (i summed):   → (B, S)."""
    return -mu + torch.einsum("bi,ij->bj", pi, A) + dln_ntot[:, None]


def enthalpy(n: torch.Tensor, T: torch.Tensor, th: dict) -> torch.Tensor:
    r"""H_b = n_{bj} [ΔfH°_j + c_{p,j}(T_b - T0)]   (j summed), in kJ:   (B, S), (B,) → (B,)."""
    h = th["dH"] + 1e-3 * th["cp"] * (T[:, None] - T0)  # (B, S) kJ/mol
    return torch.einsum("bj,bj->b", n, h)


def mole_fractions(n: torch.Tensor) -> torch.Tensor:
    """x_{bj} = n_{bj} / n_tot,b."""
    return n / n.sum(-1, keepdim=True)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/gibbs_minimisation/tensors.py
"""Build the concrete tensors: species, the element (formula) matrix, the thermochemical table and the feed.

Thermochemistry (all at the standard pressure P° = 1 bar):

* ΔfH°(298.15 K) in kJ/mol and S°(298.15 K) in J/(mol K): rounded values from the NIST-JANAF thermochemical
  tables (Chase, 4th ed., 1998).
* c_p: ILLUSTRATIVE constant heat capacities, roughly the mean of c_p(T) between 298 K and 2000 K. Real c_p rises
  with T, so H(T), S(T) and the compositions below are approximate (semi-quantitative). For real work use the
  NASA 7-coefficient polynomials (e.g. from GRI-Mech 3.0 or Cantera).
"""

import torch

T0 = 298.15  # K, reference temperature of the table
R = 8.314462618  # J/(mol K)

SPECIES = ["CO2", "H2O", "CO", "H2", "O2", "N2", "OH", "H", "O", "NO"]
ELEMENTS = ["C", "H", "O", "N"]

# species: (atoms of C, H, O, N), ΔfH°298 (kJ/mol), S°298 (J/mol/K), c_p (J/mol/K, illustrative constant)
TABLE = {
    "CO2": ((1, 0, 2, 0), -393.52, 213.79, 53.7),
    "H2O": ((0, 2, 1, 0), -241.83, 188.83, 42.7),
    "CO": ((1, 0, 1, 0), -110.53, 197.66, 33.3),
    "H2": ((0, 2, 0, 0), 0.0, 130.68, 31.1),
    "O2": ((0, 0, 2, 0), 0.0, 205.15, 34.8),
    "N2": ((0, 0, 0, 2), 0.0, 191.61, 33.0),
    "OH": ((0, 1, 1, 0), 38.99, 183.71, 30.2),
    "H": ((0, 1, 0, 0), 218.00, 114.72, 20.8),
    "O": ((0, 0, 1, 0), 249.17, 161.06, 21.0),
    "NO": ((0, 0, 1, 1), 90.29, 210.76, 34.0),
}
DH_CH4 = -74.87  # kJ/mol, ΔfH°298 of methane (NIST-JANAF): the fuel enters at 298 K
N2_PER_O2 = 3.76  # dry air: 79 % N2 / 21 % O2 (argon counted as nitrogen)


def element_matrix(device: torch.device) -> torch.Tensor:
    """A_{ij} = number of atoms of element i in species j, shape (E, S) = (4, 10)."""
    rows = [TABLE[s][0] for s in SPECIES]  # (S, E)
    return torch.tensor(rows, dtype=torch.float64, device=device).T


def thermo(device: torch.device) -> dict:
    """ΔfH°298 (kJ/mol), S°298 (J/mol/K) and c_p (J/mol/K), each of shape (S,)."""

    def column(k):
        return torch.tensor([TABLE[s][k] for s in SPECIES], dtype=torch.float64, device=device)

    return {"dH": column(1), "S": column(2), "cp": column(3)}


def feed(phi: torch.Tensor) -> torch.Tensor:
    """Element totals b_{fi} for CH4 + (2/φ)(O2 + 3.76 N2), one row per equivalence ratio φ_f: (F,) → (F, 4).

    φ = 1 is stoichiometric (CH4 + 2 O2 → CO2 + 2 H2O), φ < 1 lean (excess air), φ > 1 rich.
    """
    o2 = 2.0 / phi  # mol O2 per mol CH4
    return torch.stack([torch.ones_like(phi), 4.0 * torch.ones_like(phi), 2.0 * o2, 2.0 * N2_PER_O2 * o2], dim=-1)


def reactant_enthalpy(phi: torch.Tensor) -> torch.Tensor:
    """H of the cold reactants at 298 K in kJ per mol CH4: only CH4 has a non-zero ΔfH (O2, N2 are elements)."""
    return torch.full_like(phi, DH_CH4)


def temperatures(t_min: float, t_max: float, n: int, device: torch.device) -> torch.Tensor:
    """The batch of temperatures T_b (K), shape (B,)."""
    return torch.linspace(t_min, t_max, n, dtype=torch.float64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/gibbs_minimisation/solve.py
"""Batched Newton iteration for the Gibbs minimum (Gordon–McBride / RAND), and the adiabatic flame temperature."""

import math

import torch

from problems._common.progress import steps

from .equation import (
    chemical_potential_log,
    element_totals,
    enthalpy,
    newton_system,
    species_update,
    standard_potential,
)
from .tensors import T0

TRACE = math.log(1e-8)  # species below this mole fraction count as "trace" for the step control


def equilibrium(g: torch.Tensor, A: torch.Tensor, b: torch.Tensor, P: float, n_iter: int = 60, start=None) -> dict:
    """Minimise G/RT for every row of the batch at once.   g (B, S), A (E, S), b (B, E), P in bar.

    Each Newton step solves one (E+1)×(E+1) system per row (torch.linalg.solve on a (B, E+1, E+1) batch) for
    the element potentials π and Δln n_tot, then updates every ln n_j. The step is damped as in NASA's CEA
    code: majors may change by at most a factor e² per step, and trace species may not jump above 1e-4.
    Returns n (B, S), the element potentials pi (B, E) and the last step size.
    """
    B, S = g.shape
    if start is None:  # all species equally abundant, n_tot = Σ b / 2
        ln_ntot = torch.log(b.sum(-1) / 2)
        ln_n = (ln_ntot - math.log(S))[:, None].expand(B, S).clone()
    else:
        ln_n, ln_ntot = start["ln_n"].clone(), start["ln_ntot"].clone()

    for _ in steps(n_iter, "Newton steps"):
        n, n_tot = torch.exp(ln_n), torch.exp(ln_ntot)
        mu = chemical_potential_log(ln_n, ln_ntot, g, P)  # (B, S)
        J, rhs = newton_system(A, n, n_tot, mu, b)
        sol = torch.linalg.solve(J, rhs)  # (B, E+1)
        pi, dln_ntot = sol[:, :-1], sol[:, -1]
        dln_n = species_update(A, mu, pi, dln_ntot)  # (B, S)

        # step control: λ ≤ 1, majors change by ≤ e², trace species stop at mole fraction 1e-4
        major = (ln_n - ln_ntot[:, None]) > TRACE
        big = torch.where(major, dln_n.abs(), torch.zeros_like(dln_n)).amax(-1)
        lam1 = 2.0 / torch.maximum(5 * dln_ntot.abs(), big).clamp_min(1e-300)
        rise = dln_n - dln_ntot[:, None]
        cap = torch.where(~major & (rise > 0), (-(ln_n - ln_ntot[:, None]) - 9.2103) / rise, torch.inf)
        lam = torch.minimum(torch.minimum(lam1, cap.abs().amin(-1)), torch.ones_like(lam1))
        ln_n = ln_n + lam[:, None] * dln_n
        ln_ntot = ln_ntot + lam * dln_ntot

    return {"n": torch.exp(ln_n), "pi": pi, "ln_n": ln_n, "ln_ntot": ln_ntot, "step": (lam[:, None] * dln_n).abs()}


def temperature_sweep(T: torch.Tensor, A: torch.Tensor, b: torch.Tensor, P: float, th: dict, n_iter: int = 60):
    """Equilibrium composition at every temperature of the batch T (B,) for ONE feed b (E,)."""
    g = standard_potential(T, th)  # (B, S)
    return equilibrium(g, A, b.expand(len(T), -1), P, n_iter)


def flame_temperature(A, b, H_react, P, th, n_bisect: int = 26, t_lo: float = 900.0, t_hi: float = 3600.0):
    """Adiabatic flame temperature for every feed row of b (F, E): find T with H_products(T, n_eq(T)) = H_reactants.

    Bisection, all F feeds in parallel; each step warm-starts Newton from the previous composition.
    """
    lo = torch.full_like(H_react, t_lo)
    hi = torch.full_like(H_react, t_hi)
    state = None
    for _ in steps(n_bisect, "bisection"):
        mid = 0.5 * (lo + hi)
        state = equilibrium(standard_potential(mid, th), A, b, P, n_iter=60 if state is None else 12, start=state)
        too_hot = enthalpy(state["n"], mid, th) > H_react  # products hold more enthalpy than was released
        hi = torch.where(too_hot, mid, hi)
        lo = torch.where(too_hot, lo, mid)
    return 0.5 * (lo + hi), state["n"]


def frozen_flame_temperature(n_cold: torch.Tensor, H_react: torch.Tensor, th: dict) -> torch.Tensor:
    """T_ad if the products kept a fixed (cold, undissociated) composition n_cold (F, S): H is linear in T,

    H_react = n_j ΔfH_j + n_j c_pj (T - T0)   ⇒   T = T0 + (H_react - n_j ΔfH_j) / (n_j c_pj).
    """
    h_form = torch.einsum("fj,j->f", n_cold, th["dH"])  # kJ
    heat_capacity = 1e-3 * torch.einsum("fj,j->f", n_cold, th["cp"])  # kJ/K
    return T0 + (H_react - h_form) / heat_capacity


def residuals(A: torch.Tensor, b: torch.Tensor, n: torch.Tensor, pi: torch.Tensor, g: torch.Tensor, P: float):
    """Largest violations of the two optimality conditions: element balance and μ_j/RT = π_i A_ij (majors only)."""
    balance = (element_totals(A, n) - b).abs().max().item()
    ln_n, ln_ntot = torch.log(n), torch.log(n.sum(-1))
    mu = chemical_potential_log(ln_n, ln_ntot, g, P)
    major = (ln_n - ln_ntot[:, None]) > TRACE
    kkt = (mu - torch.einsum("bi,ij->bj", pi, A))[major].abs().max().item()
    return balance, kkt


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/gibbs_minimisation/plot.py
"""Animate a temperature sweep: equilibrium mole fractions x_j(T), the composition at the cursor, and T_ad(φ)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

X_MIN = 1e-6  # lower edge of the log axes
COLORS = plt.get_cmap("tab10").colors
LABELS = {"CO2": "CO₂", "H2O": "H₂O", "H2": "H₂", "O2": "O₂", "N2": "N₂"}


def animate(
    T: torch.Tensor,
    x: torch.Tensor,
    G: torch.Tensor,
    phis: torch.Tensor,
    T_ad: torch.Tensor,
    T_frozen: torch.Tensor,
    phi0: float,
    T_ad0: float,
    names: list[str],
    path: Path,
) -> Path:
    t, xs, gs = T.cpu().numpy(), x.cpu().numpy(), G.cpu().numpy()  # (B,), (B, S), (B,)
    ph, ta, tf = phis.cpu().numpy(), T_ad.cpu().numpy(), T_frozen.cpu().numpy()
    labels = [LABELS.get(s, s) for s in names]
    S = len(names)

    fig = plt.figure(figsize=(8.2, 4.4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.35, 1], height_ratios=[1.2, 1])
    ax_x = fig.add_subplot(grid[:, 0])
    ax_b = fig.add_subplot(grid[0, 1])
    ax_t = fig.add_subplot(grid[1, 1])

    # --- mole fractions against temperature ---
    for j in range(S):
        ax_x.plot(t, np.clip(xs[:, j], X_MIN / 10, 1), color=COLORS[j], lw=1.6)
    for j, y_label in _spread(xs[-1]).items():  # label each curve at the hot end, without overlaps
        ax_x.text(t[-1] + 15, y_label, labels[j], color=COLORS[j], fontsize=7, va="center")
    ax_x.set_yscale("log")
    ax_x.minorticks_off()
    ax_x.set_ylim(X_MIN, 1.5)
    ax_x.set_xlim(t[0], t[-1] + 180)
    ax_x.set_xlabel("temperature T (K)")
    ax_x.set_ylabel("equilibrium mole fraction $x_j$")
    ax_x.set_title(f"CH₄ + air products, φ = {phi0:.2f}", fontsize=10)
    if t[0] < T_ad0 < t[-1]:
        ax_x.axvline(T_ad0, color="#f59e0b", ls="--", lw=1.2)
        ax_x.text(T_ad0 - 20, 2.2 * X_MIN, f"$T_{{ad}}$ = {T_ad0:.0f} K", color="#b45309", fontsize=8, ha="right")
    cursor = ax_x.axvline(t[0], color="#374151", lw=1)

    # --- composition at the cursor temperature ---
    y = np.arange(S)[::-1]
    bars = ax_b.barh(y, np.full(S, X_MIN), left=X_MIN, color=COLORS[:S], height=0.7)
    ax_b.set_xscale("log")
    ax_b.minorticks_off()
    ax_b.set_xlim(X_MIN, 1.5)
    ax_b.set_yticks(y, labels, fontsize=7)
    ax_b.tick_params(axis="x", labelsize=7)
    ax_b.grid(axis="y", visible=False)
    title = ax_b.set_title("", fontsize=9)

    # --- adiabatic flame temperature against equivalence ratio ---
    ax_t.plot(ph, tf, "--", color="#9ca3af", lw=1.4, label="no dissociation")
    ax_t.plot(ph, ta, color="#dc2626", lw=1.8, label="equilibrium")
    ax_t.plot([phi0], [T_ad0], "o", color="#dc2626", ms=5)
    ax_t.set_xlabel("equivalence ratio φ", fontsize=8, labelpad=1)
    ax_t.set_ylabel("$T_{ad}$ (K)", fontsize=8)
    ax_t.tick_params(labelsize=7)
    ax_t.set_title("adiabatic flame temperature", fontsize=9)
    ax_t.legend(fontsize=7, loc="lower center")

    def update(i: int):
        cursor.set_xdata([t[i], t[i]])
        for j, bar in enumerate(bars):
            bar.set_width(max(xs[i, j], X_MIN) - X_MIN)
        title.set_text(f"T = {t[i]:.0f} K   ·   G/RT = {gs[i]:.1f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 60), path, fps=10)


def _spread(values: np.ndarray, gap: float = 0.17) -> dict[int, float]:
    """Label heights for the visible curves: start at each curve's end and push neighbours apart (both ways)
    until they are at least `gap` decades apart on the log axis."""
    order = [j for j in np.argsort(values) if values[j] > X_MIN]
    h = np.log10(values[order])
    for _ in range(200):
        for k in range(len(h) - 1):
            overlap = gap - (h[k + 1] - h[k])
            if overlap > 0:
                h[k] -= overlap / 2
                h[k + 1] += overlap / 2
    h -= max(0.0, h.max() - 0.1)  # keep the highest label inside the axes
    return {j: 10**hk for j, hk in zip(order, h)}


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'phi': 1.0,
    'P_bar': 1.0,
    'T_max': 3500.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.gibbs_minimisation")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.gibbs_minimisation.main")
problem_dir = Path("problems/chemistry/gibbs_minimisation")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
